# Copper V25 — Fairness Audit & Missing Walk-forward (Kaggle Run All)

**Scope:** FS06 = `Copper, Aluminium, Tin, WTI`; 542 common out-of-sample observations (80/20), refit every 63 sessions, seeds 41–45. This notebook trains **only missing fitted-model architectures** for expanding walk-forward; it does **not** rerun V23's `lstm_residual`, `gru_residual` or `reopt_sa_lstm`.

- 9 missing families: LSTM/GRU/CNN-LSTM/CNN-GRU **Direct**, SimpleRNN Direct, CNN-LSTM/CNN-GRU **Residual**, Ridge Residual, Paper SA-LSTM.
- Default: run all missing expanding models; 9 folds × (8 neural × 5 seeds + 1 deterministic Ridge) = **369 fits**. Ridge predictions are shared across seeds for aligned comparison.
- Optional: turn on `RUN_STRATEGY_AUDIT` for a **targeted** walk-forward experiment with Rolling 756/Recency on three leading families (seed 42 only). These aren't a substitute for a full factorial strategy/seeds analysis.
- If V23 results ZIP is attached as Kaggle input, the notebook checks date/target alignment and produces a merged **12-model** ranking. If not, it exports V25-only results and says comparison incomplete.

**Important:** Matching a forecast calendar and retraining cadence does **not** eliminate model/HP/tuning budget differences. Report this explicitly; do not present post-hoc ranks as confirmatory evidence. No model results are fabricated.


In [ ]:
from pathlib import Path
import gc, json, hashlib, time, zipfile, warnings, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import norm
import tensorflow as tf

TARGET='Copper'
FEATURES=['Copper','Aluminium','Tin','WTI']
FEATURE_SETS={'wti_replacement_with_copper': FEATURES}
FEATURE_KEY='wti_replacement_with_copper'
LOOKBACK=22
SEED=42
BATCH_SIZE=64
RATIOS=(0.925,0.95)
SCHEMES=('rolling_756','recency_weighted')
WF_RATIO=0.80
WF_STEP=63
WF_SEEDS=(41,42,43,44,45)
RUN_STRATEGY_AUDIT=False  # Set True only if you want extra strategy comparisons (seed 42)
WF_MODELS=('lstm_direct','gru_direct','cnn_lstm_direct','cnn_gru_direct','baseline_rnn','cnn_lstm_residual','cnn_gru_residual','ridge_residual','paper_sa_lstm')
# SA HP are actual artifacts from V22, not reselected on test.
SA_HP={'units_1':47, 'units_2':127, 'learning_rate':5.0478806754686805e-05}
PAPER_HP={'units_1':39,'units_2':111,'learning_rate':5e-5}
# Use V22 training recipes for holdout; WF uses a shorter common retraining schedule.
HOLDOUT_EPOCHS=100
HOLDOUT_PATIENCE=9
WF_EPOCHS=45
WF_PATIENCE=7
RESUME=True  # For interrupted Kaggle runs, attach exported CSV or retain /kaggle/working within the same session.
OUT=Path('/kaggle/working/copper_v25_fairness') if Path('/kaggle/working').exists() else Path('copper_v25_fairness')
OUT.mkdir(parents=True,exist_ok=True)
CANDIDATES=[Path('/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
            Path('/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv'),
            Path('dataset_9_commodities_daily.csv'),Path('data/processed/dataset_9_commodities_daily.csv')]
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError:pass
print('TensorFlow',tf.__version__,'GPUs',tf.config.list_physical_devices('GPU'))
print('Outputs:',OUT.resolve())
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
import platform


In [ ]:
data_path=next((p for p in CANDIDATES if p.is_file()),None)
if data_path is None and Path('/kaggle/input').exists():
    hits=list(Path('/kaggle/input').rglob('dataset_9_commodities_daily.csv'))
    if len(hits)==1:data_path=hits[0]
    elif len(hits)>1:raise RuntimeError(f'Multiple dataset candidates: {hits}. Specify path to ensure correct source.')
if data_path is None:raise FileNotFoundError('Attach dataset_9_commodities_daily.csv in Kaggle Add Input.')
raw=pd.read_csv(data_path)
raw['Date']=pd.to_datetime(raw['Date'],errors='raise')
if raw.Date.duplicated().any():raise ValueError('Duplicate Date in input')
for c in FEATURES:raw[c]=pd.to_numeric(raw[c],errors='raise')
raw=raw.sort_values('Date').reset_index(drop=True)
if raw[FEATURES].isna().any().any():raise ValueError('Missing in V23 required features; verify alignment to V22.')
if not np.isfinite(raw[FEATURES].to_numpy()).all() or (raw[FEATURES]<=0).any().any():raise ValueError('Invalid prices')
data=raw.copy()
SHA=hashlib.sha256(data_path.read_bytes()).hexdigest()
EXPECTED_SHA='9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600'
if SHA!=EXPECTED_SHA:raise ValueError(f'Dataset fingerprint differs from historical V20/V21: {SHA}. Do not merge experiments with mismatched dataset!')
assert len(data)==2710, f'Unexpected row count {len(data)}'
print('Dataset verified',len(data),'rows',str(data.Date.iloc[0].date()),'to',str(data.Date.iloc[-1].date()),SHA)

## Model definitions and faithful preprocessing
All models use the same input tensor shape (22,4), scaler fit on the training region before the internal validation slice, and test timestamps. The full training recipes are inherited from V22/V23.

In [ ]:
PAPER_SA_LSTM_HP = {'units_1': 39, 'units_2': 111, 'learning_rate': 5e-5}

def build_baseline_lstm(input_shape, hp=None):
    """Mô hình LSTM chuẩn không tối ưu SA (Protocol V20 baseline)."""
    model = tf.keras.Sequential(name='Baseline_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

def build_paper_sa_lstm(input_shape, hp=None):
    """Kiến trúc SA-LSTM với siêu tham số công bố trong bài báo Chen et al. (2023)."""
    model = tf.keras.Sequential(name='Paper_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(39, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(111, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='mse')
    return model

def build_sa_lstm(input_shape, hp=None):
    """Kiến trúc SA-LSTM nhận siêu tham số linh hoạt do SA tìm kiếm (chỉ H=1, HP tìm mới riêng cho từng feature set)."""
    h = hp or PAPER_SA_LSTM_HP
    u1 = int(h.get('units_1', 39))
    u2 = int(h.get('units_2', 111))
    lr = float(h.get('learning_rate', 5e-5))

    model = tf.keras.Sequential(name='Reoptimized_SA_LSTM')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(u1, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(u2, activation='tanh', recurrent_activation='sigmoid'))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse')
    return model

def build_baseline_rnn(input_shape, hp=None):
    """SimpleRNN baseline with two recurrent layers and comparable head."""
    model = tf.keras.Sequential(name='Baseline_RNN')
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.SimpleRNN(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.SimpleRNN(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    return model

In [ ]:
def reset_keras(seed=SEED):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(int(seed))

def sample_weights(length, scheme, half_life=252):
    if scheme != 'recency_weighted': return None
    ages = np.arange(length - 1, -1, -1, dtype='float64')
    w = np.exp(-np.log(2) * ages / half_life)
    return (w / w.mean()).astype('float32')

def evaluate_predictions(ctx, model_name, p_val, p_test, seed=SEED, extras=None):
    ysc = ctx['ysc']; va = ctx['val']; te = ctx['test']
    def original(z): return ysc.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel()

    # 1. Dịch ngược về giá gốc (USD)
    y_true = original(te['y'])
    pred = original(p_test)
    base = original(te['prev'])

    delta_pred = pred - base
    delta_true = y_true - base

    # 2. Tính các độ đo trên giá gốc (RAW - USD) khớp 100% với V20
    mae = float(mean_absolute_error(y_true, pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, pred)))
    mse = float(mean_squared_error(y_true, pred))  # DỊCH NGƯỢC LẠI GIÁ GỐC (USD^2) - KHỚP V20
    base_mae = float(mean_absolute_error(y_true, base))
    improvement = float(100.0 * (base_mae - mae) / base_mae) if base_mae > 0 else 0.0

    # 3. Tính các độ đo trên thang đo chuẩn hóa (SCALED [0, 1])
    val_mse_scaled = float(mean_squared_error(va['y'], p_val))
    mse_scaled = float(mean_squared_error(te['y'], p_test)) if p_test is not None else np.nan

    # 4. Tính Directional Accuracy và các thống kê kiểm toán phần dư
    dir_acc = float(100.0 * np.mean(np.sign(delta_pred) == np.sign(delta_true))) if model_name != 'naive' else np.nan
    delta_pred_std = float(np.std(delta_pred))
    delta_true_std = float(np.std(delta_true))
    delta_pred_mean_abs = float(np.mean(np.abs(delta_pred)))
    pred_delta_corr = float(np.corrcoef(delta_pred, delta_true)[0, 1]) if delta_pred_std > 1e-12 else np.nan

    row = {
        'model': model_name,
        'feature_set': ctx['feature_set'],
        'copper_history': 'Copper' in ctx['features'],
        'feature_group': ctx['feature_set'].split('_')[0],
        'scheme': ctx['scheme'],
        'ratio': ctx['ratio'],
        'seed': int(seed),
        'n_test': len(y_true),
        'mae': mae,
        'rmse': rmse,
        'mse': mse,                        # GIÁ GỐC DỊCH NGƯỢC (USD^2) - KHỚP V20
        'mse_scaled': mse_scaled,          # SCALED TRÊN TẬP TEST [0, 1]
        'mae_naive': base_mae,
        'mae_improvement_pct': improvement,
        'val_mse_scaled': val_mse_scaled,  # SCALED TRÊN TẬP VALIDATION [0, 1] - KHỚP V20
        'directional_accuracy_pct': dir_acc,
        'delta_pred_std_raw': delta_pred_std,
        'delta_true_std_raw': delta_true_std,
        'delta_pred_mean_abs_raw': delta_pred_mean_abs,
        'pred_delta_corr': pred_delta_corr,
        'test_start': str(pd.Timestamp(te['date'][0]).date()),
        'fit_start': str(data.Date.iloc[ctx['fit_start']].date())
    }
    if extras: row.update(extras)

    predictions = pd.DataFrame({
        'Date': te['date'],
        'ratio': ctx['ratio'],
        'scheme': ctx['scheme'],
        'feature_set': ctx['feature_set'],
        'copper_history': 'Copper' in ctx['features'],
        'feature_group': ctx['feature_set'].split('_')[0],
        'model': model_name,
        'seed': int(seed),
        'actual': y_true,
        'pred': pred,
        'naive': base,
        'pred_delta': delta_pred,
        'actual_delta': delta_true
    })
    return row, predictions

def fit_predict_model(builder, ctx, hp=None, seed=SEED, epochs=None, patience=None, is_search=False, skip_test=False):
    reset_keras(seed)
    tr, va, te = ctx['train'], ctx['val'], ctx['test']
    model = builder(tr['X'].shape[1:], hp)

    cb = []
    if CFG['earlystop']:
        pat = patience if patience is not None else (4 if is_search else 9)
        cb.append(tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=pat, restore_best_weights=True))

    ep = epochs if epochs is not None else (CFG['search_epochs'] if is_search else CFG['epochs'])
    weights = sample_weights(len(tr['idx']), ctx['scheme'])

    start_t = time.perf_counter()
    hist = model.fit(
        tr['X'], tr['y'],
        validation_data=(va['X'], va['y']),
        epochs=ep,
        batch_size=BATCH_SIZE,
        sample_weight=weights,
        callbacks=cb,
        shuffle=False,
        verbose=0
    )
    fit_sec = time.perf_counter() - start_t

    pv = model.predict(va['X'], verbose=0).ravel()
    pt = model.predict(te['X'], verbose=0).ravel() if not skip_test else None
    val_mse = float(mean_squared_error(va['y'], pv))
    epochs_run = len(hist.history['loss'])

    del model
    reset_keras(seed)
    return {'val_pred': pv, 'test_pred': pt, 'val_mse': val_mse, 'seconds': fit_sec, 'epochs_run': epochs_run}

In [ ]:
def build_net(arch, shape):
    inp=tf.keras.Input(shape=shape)
    z=inp
    if arch in ('cnn_lstm','cnn_gru'):
        z=tf.keras.layers.Conv1D(32,3,padding='causal',activation='relu')(z)
        z=tf.keras.layers.MaxPooling1D(2)(z)
    if arch in ('lstm','cnn_lstm'):
        n1,n2=(64,32) if arch=='lstm' else (48,24)
        z=tf.keras.layers.LSTM(n1,return_sequences=True)(z)
        z=tf.keras.layers.Dropout(.2)(z)
        z=tf.keras.layers.LSTM(n2)(z)
    elif arch in ('gru','cnn_gru'):
        n1,n2=(64,32) if arch=='gru' else (48,24)
        z=tf.keras.layers.GRU(n1,return_sequences=True)(z)
        z=tf.keras.layers.Dropout(.2)(z)
        z=tf.keras.layers.GRU(n2)(z)
    else: raise ValueError(arch)
    z=tf.keras.layers.Dropout(.2)(z)
    z=tf.keras.layers.Dense(16,activation='relu')(z)
    out=tf.keras.layers.Dense(1,activation='linear')(z)
    model=tf.keras.Model(inp,out,name=arch)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='mse')
    return model

In [ ]:
def fold_context(origin,end):
    val_start=int(origin*0.9)
    assert LOOKBACK+30<val_start<origin<end<=len(data)
    xsc=MinMaxScaler().fit(data.iloc[:val_start][FEATURES])
    ysc=MinMaxScaler().fit(data.iloc[:val_start][['Copper']])
    x=xsc.transform(data[FEATURES]).astype('float32')
    y=ysc.transform(data[['Copper']]).ravel().astype('float32')
    prev=np.r_[np.nan,y[:-1]]
    def block(first,last):
        ids=np.arange(max(first,LOOKBACK),last)
        return {'idx':ids,'X':np.stack([x[t-LOOKBACK:t] for t in ids]),'y':y[ids],
                'prev':prev[ids].astype('float32'),'date':data.Date.iloc[ids].to_numpy()}
    tr=block(LOOKBACK,val_start);va=block(val_start,origin);te=block(origin,end)
    assert tr['idx'][-1]<va['idx'][0]<te['idx'][0]
    assert np.allclose(te['X'][:,-1,0],te['prev'],atol=2e-6)
    return {'train':tr,'val':va,'test':te,'xsc':xsc,'ysc':ysc,'features':FEATURES,
            'feature_set':FEATURE_KEY,'scheme':'expanding','ratio':origin/len(data),
            'fit_start':0,'val_start':val_start,'test_start':origin}



## Fairness and leakage checks

Forecasts use only information available up through `t-1` for each test target `t`; scaler is fitted on subtrain, early stopping uses historical validation before test. The 63-session updates happen only at fold boundaries. A separate audit later verifies all models predict the same 542 dates and share the same target values.

In [ ]:

def build_simple_rnn(shape):
    return build_baseline_rnn(shape)

def run_missing_fold(ctx,name,seed):
    tr,va,te=[ctx[k] for k in ('train','val','test')]
    if name=='ridge_residual':
        t0=time.monotonic()
        model=Ridge(alpha=1.0)
        model.fit(tr['X'].reshape(len(tr['X']),-1),tr['y']-tr['prev'])
        pv=va['prev']+model.predict(va['X'].reshape(len(va['X']),-1))
        pt=te['prev']+model.predict(te['X'].reshape(len(te['X']),-1))
        return pv,pt,{'epochs_run':0,'fit_seconds':time.monotonic()-t0,'hp_config':'Ridge alpha=1'}
    reset_keras(seed)
    if name=='paper_sa_lstm':
        model=build_paper_sa_lstm(tr['X'].shape[1:],PAPER_HP)
    elif name=='baseline_rnn':
        model=build_simple_rnn(tr['X'].shape[1:])
    else:
        arch=name.rsplit('_',1)[0]
        model=build_net(arch,tr['X'].shape[1:])
    residual=name.endswith('_residual')
    train_y=tr['y']-tr['prev'] if residual else tr['y']
    val_y=va['y']-va['prev'] if residual else va['y']
    cb=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=WF_PATIENCE,restore_best_weights=True)]
    t0=time.monotonic()
    hist=model.fit(tr['X'],train_y,validation_data=(va['X'],val_y),
                   batch_size=BATCH_SIZE,epochs=WF_EPOCHS,callbacks=cb,
                   shuffle=False,verbose=0)
    pv=model.predict(va['X'],verbose=0).ravel()
    pt=model.predict(te['X'],verbose=0).ravel()
    if residual:
        pv=pv+va['prev'];pt=pt+te['prev']
    hp_info=('Paper HP 39/111 LR=5e-5' if name=='paper_sa_lstm' else 'fixed V22')
    meta={'epochs_run':len(hist.history['loss']),
          'fit_seconds':time.monotonic()-t0, 'hp_config':hp_info}
    del model
    reset_keras(seed)
    return pv,pt,meta

def append_frame(path,frame):
    frame.to_csv(path,mode='a',header=not path.exists(),index=False)

def load_checked(path,run_hash):
    if not path.exists(): return pd.DataFrame()
    frame=pd.read_csv(path)
    if not frame.empty and (('run_hash' not in frame) or not frame.run_hash.eq(run_hash).all()):
        raise RuntimeError(f'Checkpoint from different configuration: {path}')
    return frame

FIRST=int(len(data)*WF_RATIO)
FOLDS=[(i,min(i+WF_STEP,len(data))) for i in range(FIRST,len(data),WF_STEP)]
assert len(FOLDS)==9 and sum(e-s for s,e in FOLDS)==542

# Verify reproducible calendar, no unseen Copper(t) in last input window.
for start,end in FOLDS:
    c=fold_context(start,end)
    assert c['train']['idx'].max()<c['val']['idx'].min()<c['test']['idx'].min()
    assert np.array_equal(c['test']['idx'],np.arange(start,end))
    assert np.allclose(c['test']['X'][:,-1,0],c['test']['prev'],atol=2e-6)
    assert c['xsc'].n_samples_seen_==c['val_start']
print('Leakage/timestamp checks passed:',len(FOLDS),'folds')


## Part A — Missing models on common expanding walk-forward
Checkpointing is per `(fold_start, seed, model)`; Ridge is fitted only once per fold. All fitted models use the same training/validation/test boundaries in each fold.

In [ ]:

RUN_HASH=hashlib.sha256(json.dumps({'sha':SHA,'fs':FEATURES,'models':WF_MODELS,'ratio':WF_RATIO,
    'step':WF_STEP,'seeds':WF_SEEDS,'epochs':WF_EPOCHS,'patience':WF_PATIENCE,
    'tf':tf.__version__,'ridge_alpha':1},sort_keys=True).encode()).hexdigest()[:16]
METRICS=OUT/'wf_missing_fold_metrics.csv'
PREDS=OUT/'wf_missing_predictions.csv'
if not RESUME:
    for f in (METRICS,PREDS):f.unlink(missing_ok=True)
old=load_checked(METRICS,RUN_HASH)
done=set(zip(old.fold_start.astype(int),old.seed.astype(int),old.model)) if len(old) else set()
expected=sum(1 if m=='ridge_residual' else len(WF_SEEDS) for m in WF_MODELS)*len(FOLDS)
print(f'Running {expected} missing fits; {len(done)} completed checkpoints')
for fi,(start,end) in enumerate(FOLDS,1):
    ctx=fold_context(start,end)
    for name in WF_MODELS:
        seeds=(WF_SEEDS[0],) if name=='ridge_residual' else WF_SEEDS
        for seed in seeds:
            k=(start,seed,name)
            if k in done:continue
            pv,pt,meta=run_missing_fold(ctx,name,seed)
            row,pred=evaluate_predictions(ctx,name,pv,pt,seed=seed,extras=meta)
            row.update({'fold_start':start,'fold_end':end,'fold_number':fi,
                        'run_hash':RUN_HASH,'source':'V25_missing_wf'})
            pred['fold_start']=start;pred['fold_end']=end;pred['fold_number']=fi
            pred['run_hash']=RUN_HASH;pred['source']='V25_missing_wf'
            append_frame(METRICS,pd.DataFrame([row]))
            append_frame(PREDS,pred)
            done.add(k)
            print(f'{len(done)}/{expected} {fi}/{len(FOLDS)} {name} seed={seed} MAE={row["mae"]:.6f}',flush=True)
assert len(done)==expected,(len(done),expected)
assert len(pd.read_csv(METRICS))==expected
print('Missing models trained:',expected,'fits')


## Part B — Optional targeted walk-forward for strategy fairness

To test whether **Expanding / Rolling 756 / Recency** changes performance when run as *walk-forward strategies* (rather than one-shot holdout), set `RUN_STRATEGY_AUDIT=True` in configuration and rerun. Only `gru_residual`, `lstm_residual`, `paper_sa_lstm`, seed 42, all three strategies, same 542 timestamps. **This is an exploratory strategy sensitivity analysis, not a full multi-seed factorial.**


In [ ]:

def fold_context_strategy(start,end,scheme):
    n=len(data); v=int(start*0.9)
    fitstart=max(0,v-756) if scheme=='rolling_756' else 0
    cols=FEATURES
    xsc=MinMaxScaler().fit(data.iloc[fitstart:v][cols])
    ysc=MinMaxScaler().fit(data.iloc[fitstart:v][['Copper']])
    x=xsc.transform(data[cols]).astype('float32')
    y=ysc.transform(data[['Copper']]).ravel().astype('float32')
    prev=np.r_[np.nan,y[:-1]]
    def block(a,b,restrict=False):
        ids=np.arange(max(a,LOOKBACK),b)
        if restrict: ids=ids[ids-LOOKBACK>=fitstart]
        return {'idx':ids,'X':np.stack([x[t-LOOKBACK:t] for t in ids]),
                'y':y[ids],'prev':prev[ids].astype('float32'),
                'date':data.Date.iloc[ids].to_numpy()}
    return {'train':block(fitstart,v,True),'val':block(v,start),
            'test':block(start,end),'xsc':xsc,'ysc':ysc,
            'features':cols,'feature_set':FEATURE_KEY,'scheme':scheme,
            'ratio':start/n,'fit_start':fitstart,'val_start':v,'test_start':start}

def strategy_fit(ctx,name,seed):
    reset_keras(seed)
    tr,va,te=[ctx[k] for k in ('train','val','test')]
    if name=='paper_sa_lstm':model=build_paper_sa_lstm(tr['X'].shape[1:],PAPER_HP)
    else:model=build_net(name.split('_')[0],tr['X'].shape[1:])
    residual=name.endswith('_residual')
    target=tr['y']-tr['prev'] if residual else tr['y']
    valtarget=va['y']-va['prev'] if residual else va['y']
    hist=model.fit(tr['X'],target,validation_data=(va['X'],valtarget),
             sample_weight=sample_weights(len(tr['idx']),ctx['scheme']),
             shuffle=False,epochs=WF_EPOCHS,batch_size=BATCH_SIZE,verbose=0,
             callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=WF_PATIENCE,restore_best_weights=True)])
    pv=model.predict(va['X'],verbose=0).ravel();pt=model.predict(te['X'],verbose=0).ravel()
    if residual:pv=pv+va['prev'];pt=pt+te['prev']
    del model;reset_keras(seed)
    return pv,pt,{'epochs_run':len(hist.history['loss'])}
STRATEGY_MODELS=('gru_residual','lstm_residual','paper_sa_lstm')
STRATEGIES=('expanding','rolling_756','recency_weighted')
STRATEGY_RESULTS=OUT/'strategy_sensitivity_fold_metrics.csv'
STRATEGY_PREDS=OUT/'strategy_sensitivity_predictions.csv'
S_HASH=hashlib.sha256(json.dumps({'sha':SHA,'models':STRATEGY_MODELS,'schemes':STRATEGIES,
 'seed':42,'step':WF_STEP,'epochs':WF_EPOCHS,'tf':tf.__version__},sort_keys=True).encode()).hexdigest()[:16]
if RUN_STRATEGY_AUDIT:
    s_old=load_checked(STRATEGY_RESULTS,S_HASH)
    s_done=set(zip(s_old.fold_start.astype(int),s_old.scheme,s_old.model)) if len(s_old) else set()
    for fi,(start,end) in enumerate(FOLDS,1):
        for scheme in STRATEGIES:
            ctx=fold_context_strategy(start,end,scheme)
            for name in STRATEGY_MODELS:
                if (start,scheme,name) in s_done: continue
                pv,pt,meta=strategy_fit(ctx,name,42)
                row,pred=evaluate_predictions(ctx,name,pv,pt,seed=42,extras=meta)
                row.update({'fold_start':start,'fold_end':end,'fold_number':fi,
                            'run_hash':S_HASH,'source':'V25_strategy_audit'})
                pred['fold_start']=start;pred['fold_end']=end;pred['fold_number']=fi
                pred['run_hash']=S_HASH;pred['source']='V25_strategy_audit'
                append_frame(STRATEGY_RESULTS,pd.DataFrame([row]));append_frame(STRATEGY_PREDS,pred)
                s_done.add((start,scheme,name))
                print('Strategy',len(s_done),'/',len(FOLDS)*len(STRATEGIES)*len(STRATEGY_MODELS),name,scheme)
    assert len(s_done)==len(FOLDS)*len(STRATEGIES)*len(STRATEGY_MODELS)
else: print('Optional strategy sensitivity SKIPPED (not represented as completed).')


## Part C — Validate forecasts, calculate MAE/RMSE/MAPE/R², paired DM-HAC, optionally merge V23

V23 ZIP is optional but recommended. Attach `20 -11 - copper_v23_gapfill_results.zip` in Kaggle Add Input (or place alongside notebook). A merged ranking is produced only when V23 predictions are available and align exactly with the V25 calendar, target and Naive reference. Seed-averaged results are descriptive; folds and seeds from the same period are not independent samples.


In [ ]:

def metric_summary(preds, keys=('model','seed')):
    out=[]
    for key,g in preds.groupby(list(keys),dropna=False):
        a=g.actual.to_numpy(dtype=float);p=g.pred.to_numpy(dtype=float);b=g.naive.to_numpy(dtype=float)
        if np.any(a<=0):mape=np.nan
        else:mape=float(np.mean(np.abs((a-p)/a))*100)
        ybar=np.mean(a);den=np.sum((a-ybar)**2)
        r2=float(1-np.sum((a-p)**2)/den) if den>0 else np.nan
        row=dict(zip(keys,key if isinstance(key,tuple) else (key,)))
        row.update(n=len(g),mae=float(np.mean(np.abs(a-p))),rmse=float(np.sqrt(np.mean((a-p)**2))),
                   mape_pct=mape,r2=r2,naive_mae=float(np.mean(np.abs(a-b))))
        out.append(row)
    return pd.DataFrame(out)

p=pd.read_csv(PREDS,parse_dates=['Date'])
assert len(p)==542*(8*len(WF_SEEDS)+1), len(p)
assert p.groupby(['model','seed']).Date.nunique().eq(542).all()
assert p.groupby(['model','seed']).size().eq(542).all()
expected_dates=data.Date.iloc[FIRST:].reset_index(drop=True)
for _,g in p.groupby(['model','seed']):
    assert g.Date.sort_values().reset_index(drop=True).equals(expected_dates)
    assert np.allclose(g.actual.to_numpy(),data.Copper.iloc[FIRST:].to_numpy(),atol=1e-4)
    assert np.allclose(g.naive.to_numpy(),data.Copper.iloc[FIRST-1:-1].to_numpy(),atol=1e-4)

summary=metric_summary(p)
summary.to_csv(OUT/'wf_missing_seed_summary.csv',index=False)
print('V25 missing-model summary:');display(summary.sort_values('mae').head(20))

# V23 exact reference is required for a complete 12-model comparison.
v23_hits=[]
for root in [Path('/kaggle/input'),Path('/kaggle/working'),Path('.')]:
    if root.exists():
        v23_hits += [f for f in root.rglob('*.zip') if 'v23' in f.name.lower() and f.is_file()]
v23_hits=list(dict.fromkeys(v23_hits))
merged=p.copy(); v23_source=None
for path in v23_hits:
    try:
        with zipfile.ZipFile(path) as z:
            matches=[f for f in z.namelist() if f.endswith('/wf_predictions.csv')]
            if not matches:continue
            v=pd.read_csv(z.open(matches[0]),parse_dates=['Date'])
        needed={'Date','model','seed','actual','pred','naive'}
        if not needed.issubset(v):continue
        assert set(v.model.unique())=={'lstm_residual','gru_residual','reopt_sa_lstm'}
        for _,g in v.groupby(['model','seed']):
            assert len(g)==542
            assert g.Date.sort_values().reset_index(drop=True).equals(expected_dates)
            assert np.allclose(g.sort_values('Date').actual.to_numpy(),data.Copper.iloc[FIRST:].to_numpy(),atol=1e-4)
        assert sorted(v.seed.unique().tolist())==list(WF_SEEDS)
        v23_source=str(path)
        merged=pd.concat([p,v],ignore_index=True)
        break
    except Exception as exc:
        print('V23 candidate rejected:',path,repr(exc))

if v23_source:
    assert merged.groupby(['model','seed']).size().eq(542).all()
    all_summary=metric_summary(merged)
    all_summary.to_csv(OUT/'wf_all_models_seed_summary.csv',index=False)
    rank=all_summary.groupby('model',as_index=False).agg(mae_mean=('mae','mean'),mae_std=('mae','std'),
                 rmse_mean=('rmse','mean'),mape_mean=('mape_pct','mean'),r2_mean=('r2','mean'))
    rank=rank.sort_values('mae_mean')
    rank['rank_by_mae']=np.arange(1,len(rank)+1)
    rank.to_csv(OUT/'wf_all_models_ranking.csv',index=False)
    display(rank)
else:
    print('No valid V23 source. Output is V25-only; do NOT claim complete 12-model fairness.')

# DM-HAC of paired absolute errors, using 542 common dates per seed.
def dm_hac_by_date(one,baseline):
    diff=np.abs(one.actual.to_numpy()-one.pred.to_numpy())-np.abs(one.actual.to_numpy()-baseline)
    diff=np.asarray(diff,dtype=float);n=len(diff)
    if n<20:return np.nan,np.nan
    dbar=diff.mean();centered=diff-dbar
    gamma0=np.dot(centered,centered)/n
    var=gamma0
    lagmax=min(5,n-1)
    for lag in range(1,lagmax+1):
        cov=np.dot(centered[lag:],centered[:-lag])/n
        var+=2*(1-lag/(lagmax+1))*cov
    if var<=0:return np.nan,np.nan
    z=dbar/np.sqrt(var/n)
    return float(z),float(2*norm.sf(abs(z)))

rows=[]
for (model,seed),g in merged.groupby(['model','seed']):
    g=g.sort_values('Date')
    z,pval=dm_hac_by_date(g,g.naive.to_numpy())
    rows.append({'model':model,'seed':seed,'n':len(g),'mean_abs_loss_diff':
                 float((np.abs(g.actual-g.pred)-np.abs(g.actual-g.naive)).mean()),
                 'dm_hac_z':z,'dm_hac_p_value':pval})
pd.DataFrame(rows).to_csv(OUT/'wf_dm_hac_by_seed.csv',index=False)

# Plot MAE by fitted model (if V23 loaded, include all).
use=metric_summary(merged).groupby('model',as_index=False).mae.mean().sort_values('mae')
fig,ax=plt.subplots(figsize=(11,max(5,len(use)*.4)))
ax.barh(use.model[::-1],use.mae[::-1]);ax.set_xlabel('MAE (original Copper price units)')
ax.set_title('WTI (FS06) walk-forward — common 542 timestamps')
fig.tight_layout();fig.savefig(OUT/'wf_model_mae.png',dpi=170);plt.close(fig)


## Part D — Provenance, experiment coverage and final ZIP

The report identifies residual gaps: only FS06 is walk-forward complete for Expanding (when V23 is attached); FS02/FS04 remain untested for most architectures under the same walk-forward dates; independent final test after model selection is unavailable; no exhaustive equal-budget HP search. None of these can be fixed by filling tables without new experiments. The optional strategy study is explicitly only three architectures, one seed.


In [ ]:

manifest={'experiment':'V25 missing-model walk-forward and fairness audit','source':'V20/V21/V22/V23/V24 notebooks',
 'dataset_sha256':SHA,'dataset_rows':len(data),'dataset_start':str(data.Date.iloc[0].date()),
 'dataset_end':str(data.Date.iloc[-1].date()),'feature_set':'FS06_WTI_CU','features':FEATURES,
 'lookback':LOOKBACK,'horizon':1,'walk_forward_start':str(data.Date.iloc[FIRST].date()),
 'walk_forward_end':str(data.Date.iloc[-1].date()),'common_test_rows':len(data)-FIRST,
 'retrain_every_sessions':WF_STEP,'fold_count':len(FOLDS),'seeds':WF_SEEDS,
 'missing_models_fitted':WF_MODELS,'v23_models_to_merge':['lstm_residual','gru_residual','reopt_sa_lstm'],
 'ridge_fit_once_per_fold':True,'run_strategy_audit':RUN_STRATEGY_AUDIT,
 'v23_source':v23_source,'training_epochs_max':WF_EPOCHS,'earlystop_patience':WF_PATIENCE,
 'tensorflow_version':tf.__version__,'sklearn_version':__import__('sklearn').__version__,
 'run_hash':RUN_HASH,'note':'No all-model equal-HP-budget claim. Split/strategy/feature-grid remains partly incomplete.'}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')

audit="""# Fairness coverage after V25

- **FS06 WTI Expanding WF, 542 matched test dates, 5 neural seeds**: all 12 fitted model families if and only if V23 ZIP is supplied and `wf_all_models_ranking.csv` was generated; Ridge is deterministic and fit only once/fold.
- **Walk-forward FS06 under Rolling/Recency**: incomplete for most architectures. Optional targeted audit 3 models × 3 strategies × 1 seed is not comprehensive.
- **Walk-forward FS02/FS04**: incomplete for most architectures on matched 2024–26 dates.
- **SA search protocol**: SA HP found on a prior validation split; other architecture HP fixed, so tuning budgets aren't equal.
- **Train/validation/test**: within folds, scaler fitted using subtrain and hyperparameters held fixed, but no untouched final time period following extensive test comparisons.
- **Future-data availability**: same-date commodity alignment across markets requires exchange-time/as-of audit.
- **Statistical inference**: DM-HAC per seed uses paired daily absolute losses; multiple models/seeds are not independent, and multiplicity/selection bias should be considered.
- **Conclusion**: rank descriptively within observed settings only. Don't claim universally best model or comprehensive methodological fairness.
"""
(OUT/'FAIRNESS_AUDIT.md').write_text(audit,encoding='utf-8')
zipout=(OUT.parent/'copper_v25_fairness_results.zip')
with zipfile.ZipFile(zipout,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f'copper_v25_fairness/{f.name}')
print('ZIP:',zipout,'size MB:',round(zipout.stat().st_size/1e6,2))
print('V23 merged:',bool(v23_source),'\nAudit:',OUT/'FAIRNESS_AUDIT.md')
